# Signpost on the Typed Decisions benchmark (measurement only)

[Typed Decisions](https://huggingface.co/datasets/LocalLLaMA/typed-decisions) gives a model one piece of state (JSON) and five typed questions about it. Every answer is a probability distribution:

| Type | Meaning | Decisions on test |
|---|---|---|
| `noul` | yes / no | 600 |
| `choice` | one of N labels | 600 |
| `score` | an ordered rubric | 800 |

The test split has 400 cases (2,000 decisions) in four workflows. Reference points (accuracy): guessing 0.31, always the majority 0.47, the gold teacher's self-agreement 0.735.

This notebook **only measures**: it never trains, never uploads, and submits nothing.

1. **Signpost and Signpost one-pass as published** on the test split (the general, zero-shot mode).
2. **Input formats chosen without the test split.** A few input formats (JSON vs readable `key: value`, a yes/no statement asked as a question, option descriptions for yes/no, a longer input limit) are compared on a 200-case sample of the benchmark's **train** split. The best is then scored once on test. Choosing an input format fits no weights, but it does use the train split, and the submission mode should be stated with that in mind (see the last cell).
3. **Calibration from our own data.** A temperature fitted on Signpost's own labelled question sets (E12 and the independent set; never the benchmark), applied to the probabilities.

Scoring uses the benchmark's own code (`bench/score_with_jobs.py`), copied in Cell 3: accuracy, KL from gold, Brier, ECE (10 bins). A single T4 is enough; about 25 minutes.

In [ ]:
# Cell 1 — Setup

!pip install -q --force-reinstall --no-deps "transformers==5.17.0"

import json, math, random, statistics, sys, time, urllib.request
from collections import Counter

import pandas as pd
import torch
import transformers
from huggingface_hub import HfApi, hf_hub_download, snapshot_download

BENCH = "LocalLLaMA/typed-decisions"
MODELS = {"signpost": "skamalj/signpost", "one_pass": "skamalj/signpost-one-pass"}
RUN_MODELS = ["signpost", "one_pass"]
SELECT_CASES = 200                   # train-split cases used to choose the input format (of 1,200)
SEED = 42
OWN_DATA = "https://raw.githubusercontent.com/skamalj/llms-the-hard-way/signpost/data/modernbert_router/"
WORK = "/kaggle/working"

assert transformers.__version__ == "5.17.0", f"transformers {transformers.__version__}: restart the kernel and run from the top"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
api = HfApi()
REV = {m: api.model_info(r).sha for m, r in MODELS.items()}
REV[BENCH] = api.dataset_info(BENCH).sha
print({"torch": torch.__version__, "transformers": transformers.__version__, "device": DEVICE,
       "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None, "revisions": REV})

In [ ]:
# Cell 2 — Data: the benchmark's test and train splits; our own labelled question sets for calibration

def bench_split(split):
    f = hf_hub_download(BENCH, f"all/{split}-00000-of-00001.parquet", repo_type="dataset", revision=REV[BENCH])
    rows = pd.read_parquet(f).to_dict("records")
    for r in rows:
        r["state"], r["questions"], r["gold"] = (json.loads(r[k]) for k in ("state", "questions", "gold"))
    return rows


TEST, TRAIN = bench_split("test"), bench_split("train")
SELECT = random.Random(SEED).sample(TRAIN, SELECT_CASES)
OWN = (json.load(urllib.request.urlopen(OWN_DATA + "choice_test/items.json"))["items"]
       + json.load(urllib.request.urlopen(OWN_DATA + "independent/questions.json"))["items"])
print({"test cases": len(TEST), "test decisions": sum(len(r["gold"]) for r in TEST), "train cases": len(TRAIN),
       "selection cases": len(SELECT), "own labelled questions (calibration)": len(OWN)})
print(Counter(q["type"] for r in TEST for q in r["questions"].values()), Counter(r["workflow"] for r in TEST))

In [ ]:
# Cell 3 — The benchmark's scoring, copied from bench/score_with_jobs.py (score_one and the 10-bin ECE)

def score_one(pred, gold):
    opts = list(gold["probabilities"].keys())
    g = [float(gold["probabilities"][o]) for o in opts]
    if gold["type"] == "noul":  # engines report yes/no; gold uses true/false
        pred = dict(pred)
        pred["probabilities"] = {
            {"yes": "true", "no": "false"}.get(k, k): v for k, v in pred["probabilities"].items()
        }
    p = [float(pred["probabilities"].get(o, 0.0)) for o in opts]
    s = sum(p)
    p = [v / s for v in p] if s > 0 else [1.0 / len(opts)] * len(opts)
    am_p = p.index(max(p))
    am_g = g.index(max(g))
    kl = sum(gi * math.log(gi / max(pi, 1e-9)) for gi, pi in zip(g, p) if gi > 0)
    brier = sum((pi - gi) ** 2 for pi, gi in zip(p, g))
    return am_p == am_g, kl, brier, max(p)


def summarise(rows):
    """rows: dicts with ok, kl, brier, conf -> the benchmark's four numbers."""
    n, n_bins = len(rows), 10
    hits, total, conf = [0] * n_bins, [0] * n_bins, [0.0] * n_bins
    for r in rows:
        b = min(int(r["conf"] * n_bins), n_bins - 1)
        total[b] += 1
        hits[b] += int(r["ok"])
        conf[b] += r["conf"]
    ece = sum(total[b] / n * abs(hits[b] / total[b] - conf[b] / total[b]) for b in range(n_bins) if total[b])
    return {"n": n, "accuracy": round(sum(r["ok"] for r in rows) / n, 4), "kl_from_gold": round(sum(r["kl"] for r in rows) / n, 4),
            "brier": round(sum(r["brier"] for r in rows) / n, 4), "ece": round(ece, 4)}

In [ ]:
# Cell 4 — Turning a benchmark question into a Signpost call; the input formats compared

FORMATS = {
    "as_published": dict(state="json", noul="statement", noul_desc=False, max_len=None),
    "kv":           dict(state="kv",   noul="statement", noul_desc=False, max_len=None),
    "kv_q":         dict(state="kv",   noul="question",  noul_desc=False, max_len=None),
    "kv_q_desc":    dict(state="kv",   noul="question",  noul_desc=True,  max_len=None),
    "kv_q_desc_512": dict(state="kv",  noul="question",  noul_desc=True,  max_len=512),
}


def kv_lines(x, prefix=""):
    """Nested JSON -> readable 'key: value' lines (nested keys joined with spaces, lists joined with '; ')."""
    if isinstance(x, dict):
        return [line for k, v in x.items() for line in kv_lines(v, f"{prefix}{k.replace('_', ' ')} ")]
    if isinstance(x, list):
        if all(not isinstance(v, (dict, list)) for v in x):
            return [f"{prefix.strip()}: {'; '.join(map(str, x))}"]
        return [line for i, v in enumerate(x, 1) for line in kv_lines(v, f"{prefix}{i} ")]
    return [f"{prefix.strip()}: {x}"]


def state_text(state, fmt):
    return json.dumps(state, ensure_ascii=False) if fmt["state"] == "json" else "\n".join(kv_lines(state))


def as_question(statement):
    s = statement.strip().rstrip(".")
    return f"Is it true that {s[0].lower() + s[1:]}?"


def options_for(q, fmt):
    """-> (question text, {option shown to Signpost: description}, [gold key of each option, in order])."""
    crit = q.get("criteria") or {}
    if q["type"] == "choice":
        return q["instructions"], {k.replace("_", " "): v for k, v in crit.items()}, list(crit)
    if q["type"] == "noul":
        text = q["instructions"] if fmt["noul"] == "statement" else as_question(q["instructions"])
        desc = (lambda k: crit.get(k, "")) if fmt["noul_desc"] else (lambda k: "")
        return text, {"yes": desc("true"), "no": desc("false")}, ["true", "false"]
    # score, ordered low -> high: "Name: description" if every level has a short name, else "level 1..n: description"
    parts = [c.partition(":") for c in crit]
    named = all(sep and 0 < len(n.split()) <= 3 for n, sep, _ in parts) and len({n.strip() for n, _, _ in parts}) == len(parts)
    opts = {n.strip(): d.strip() for n, _, d in parts} if named else {f"level {i + 1}": c for i, c in enumerate(crit)}
    return q["instructions"], opts, [str(i) for i in range(len(crit))]


def load_model(key):
    path = snapshot_download(MODELS[key], revision=REV[key])
    sys.path.insert(0, path)
    if key == "signpost":
        from signpost import Signpost
        return Signpost(path, device=DEVICE)
    from signpost_one_pass import SignpostOnePass
    return SignpostOnePass(path, device=DEVICE)


def call(model, text, question, options, max_len):
    """Signpost probabilities over the options. Too long: Signpost cuts the input's start (its published behaviour);
    one-pass refuses, so the state is cut to its first N tokens until the options fit."""
    published = model.cfg["max_len"]
    model.cfg["max_len"] = max_len or published
    try:
        for keep in (None, 600, 400, 250, 150):
            t = text if keep is None else model.tok.convert_tokens_to_string(model.tok.tokenize(text)[:keep])
            try:
                return model.probs([{"role": "user", "content": t}], options, question)
            except ValueError:                                   # one-pass: options don't fit beside the state
                continue
        raise ValueError("does not fit even with a short state")
    finally:
        model.cfg["max_len"] = published


def run_cases(model, cases, fmt):
    """-> one record per decision: workflow, question id, type, gold, Signpost's probabilities by gold key."""
    out, times = [], []
    for case in cases:
        text, t0 = state_text(case["state"], fmt), time.perf_counter()
        for qid, q in case["questions"].items():
            question, opts, keys = options_for(q, fmt)
            p = call(model, text, question, opts, fmt["max_len"])
            out.append({"id": case["id"], "workflow": case["workflow"], "qid": qid, "type": q["type"], "gold": case["gold"][qid],
                        "probs": dict(zip(keys, p.values()))})
        times.append(time.perf_counter() - t0)
    return out, times


def scored(records, temperature=1.0):
    rows = []
    for r in records:
        p = {k: max(v, 1e-12) ** (1 / temperature) for k, v in r["probs"].items()}
        s = sum(p.values())
        ok, kl, brier, conf = score_one({"probabilities": {k: v / s for k, v in p.items()}}, r["gold"])
        rows.append({**r, "ok": ok, "kl": kl, "brier": brier, "conf": conf})
    return rows


def breakdown(rows):
    out = {"all": summarise(rows)}
    for field in ("type", "workflow"):
        for v in sorted({r[field] for r in rows}):
            out[f"{field}={v}"] = summarise([r for r in rows if r[field] == v])
    return out

In [ ]:
# Cell 5 — Calibration temperature, fitted on our own labelled questions only (never the benchmark)

def fit_temperature(model):
    """Negative log-likelihood of the right answer on our E12 + independent questions, over a grid of temperatures."""
    probs = []                                          # (probability of the right answer, all probabilities)
    for it in OWN:
        p = call(model, it["state"], it["question"], {o: "" for o in it["options"]}, None)
        probs.append((max(p[it["answer"]], 1e-12), [max(v, 1e-12) for v in p.values()]))
    grid = [0.5, 0.75, 1, 1.5, 2, 3, 4, 5, 6, 8, 10, 12, 15, 20]
    def nll(t):
        return statistics.mean(-math.log(pr ** (1 / t) / sum(v ** (1 / t) for v in allp)) for pr, allp in probs)
    best = min(grid, key=nll)
    acc = statistics.mean(pr == max(allp) for pr, allp in probs)
    return best, {"own_accuracy": round(acc, 3), "nll_at_1": round(nll(1), 3), "nll_at_best": round(nll(best), 3)}

In [ ]:
# Cell 6 — For each model: choose the input format on the train sample, fit the temperature, score on test

RESULTS = {}
for key in RUN_MODELS:
    model = load_model(key)
    print(f"\n===== {key} ({MODELS[key]} @ {REV[key][:8]}) =====")
    selection = {}
    for name, fmt in FORMATS.items():
        recs, _ = run_cases(model, SELECT, fmt)
        selection[name] = summarise(scored(recs))
        print(f"  train sample, {name:15s}", selection[name])
    chosen = max(FORMATS, key=lambda n: selection[n]["accuracy"])
    temperature, cal = fit_temperature(model)
    print(f"  chosen format: {chosen} | temperature from our own data: {temperature} {cal}")

    test = {}
    for name in dict.fromkeys(["as_published", chosen]):
        recs, times = run_cases(model, TEST, FORMATS[name])
        test[name] = {"T=1": breakdown(scored(recs)), f"T={temperature}": breakdown(scored(recs, temperature)),
                      "latency_s_per_case": {"p50": round(statistics.median(times), 3),
                                             "p95": round(sorted(times)[int(0.95 * len(times))], 3)},
                      "records": recs}
        print(f"  TEST {name:15s} T=1        ", test[name]["T=1"]["all"])
        print(f"  TEST {name:15s} T={temperature:<9}", test[name][f"T={temperature}"]["all"], "| latency", test[name]["latency_s_per_case"])
    RESULTS[key] = {"selection": selection, "chosen": chosen, "temperature": temperature, "calibration": cal, "test": test}
    del model
    torch.cuda.empty_cache()

In [ ]:
# Cell 7 — Results: overall, by question type and by workflow

for key, res in RESULTS.items():
    for name, t in res["test"].items():
        for temp in [k for k in t if k.startswith("T=")]:
            print(f"\n{key} | {name} | {temp} | latency per case {t['latency_s_per_case']}")
            for group, s in t[temp].items():
                print(f"  {group:38s} acc {s['accuracy']:.3f}  KL {s['kl_from_gold']:.3f}  Brier {s['brier']:.3f}  ECE {s['ece']:.3f}  (n {s['n']})")
print("\nreference points (accuracy): uniform 0.308 · majority/prior ~0.47 · teacher self-agreement 0.735")

In [ ]:
# Cell 8 — Save everything; print (never upload) the eval-results YAML the benchmark's submit script would write

summary = {"revisions": REV, "formats": FORMATS, "select_cases": SELECT_CASES,
           "models": {k: {kk: vv for kk, vv in r.items() if kk != "test"} | {"test": {n: {kk: vv for kk, vv in t.items() if kk != "records"}
                                                                                   for n, t in r["test"].items()}}
                      for k, r in RESULTS.items()}}
json.dump(summary, open(f"{WORK}/typed_decisions_summary.json", "w"), indent=1)
with open(f"{WORK}/typed_decisions_records.jsonl", "w") as f:
    for key, r in RESULTS.items():
        for name, t in r["test"].items():
            for rec in t["records"]:
                f.write(json.dumps({"model": key, "format": name, **rec}) + "\n")
print(json.dumps(summary, indent=1)[:6000])

for key, r in RESULTS.items():                     # what bench/submit_results.py --dry-run would print (as published, T=1)
    s = r["test"]["as_published"]["T=1"]["all"]
    print(f"\n# {MODELS[key]} — general mode, as published")
    for task in ("accuracy", "kl_from_gold", "brier", "ece"):
        print(f"- dataset:\n    id: {BENCH}\n    task_id: {task}\n  value: {s[task]}")

## Reading the result, and the submission mode

- **As published, T = 1** is the clean general (zero-shot) number. Nothing about the benchmark influenced it.
- **The chosen format** was picked using 200 cases of the benchmark's train split. No weights were fitted, but if in doubt, report it as a separate row with that note, or as specialist.
- **The temperature** was fitted on Signpost's own question sets only. It changes KL, Brier and ECE, not accuracy.
- Nothing here is submitted. Submitting means either opening a discussion on the dataset with the numbers and the mode, or adding a `.eval_results/` YAML (as printed above) to the model repo.